# Prediciendo diabetes


#### Paso 1: Carga del conjunto de datos

In [ ]:
import pandas as pd

url = "https://breathecode.herokuapp.com/asset/internal-link?id=930&path=diabetes.csv"
df = pd.read_csv(url)

df.head()
df.info()
df.describe()

#### Paso 2: EDA completo y división del dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df.isnull().sum()
df.duplicated().sum()

cols_with_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in cols_with_zero:
    df[col] = df[col].replace(0, np.nan)

df.fillna(df.median(), inplace=True)

plt.figure()
sns.countplot(x="Outcome", data=df)
plt.show()

plt.figure()
sns.heatmap(df.corr(), annot=True, fmt=".2f")
plt.show()

from sklearn.model_selection import train_test_split

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#### Paso 3: Modelo de árbol de decisión

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import plot_tree

criterios = ["gini", "entropy", "log_loss"]

resultados = {}

for criterio in criterios:
    model = DecisionTreeClassifier(criterion=criterio, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados[criterio] = acc

    print("Criterio:", criterio)
    print("Accuracy:", acc)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    plt.figure()
    plot_tree(model, filled=True)
    plt.show()

plt.figure()
plt.bar(resultados.keys(), resultados.values())
plt.show()

#### Paso 4: Optimización con Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy", "log_loss"]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Mejores parámetros:", grid.best_params_)
print("Accuracy final:", accuracy_score(y_test, y_pred_best))
print(confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

plt.figure()
plot_tree(best_model, filled=True)
plt.show()

### Conclusion

##### El modelo de árbol de decisión permite clasificar pacientes con diabetes utilizando variables clínicas relevantes. Tras comparar los distintos criterios de pureza, se observa que las diferencias en desempeño suelen ser pequeñas, aunque uno puede presentar mejor generalización. La optimización mediante Grid Search mejora el rendimiento al controlar la profundidad y el tamaño mínimo de los nodos, reduciendo el sobreajuste y obteniendo un modelo más robusto para predicción.